In [ ]:
import cv2
import numpy as np


def generate_random_circle(image_shape):
    center = np.random.randint(0, image_shape[0], 2)
    radius = np.random.randint(image_shape[0] // 36, image_shape[0] // 12)

    return center, radius


def check_circle(center, radius, circle_mask, colony_binary_mask):
    proposed_circle_filled = np.zeros_like(circle_mask, dtype=np.uint8)
    cv2.circle(proposed_circle_filled, center, radius, 1, -1)
    overlap = np.sum(np.logical_and(proposed_circle_filled, circle_mask))
    proposed_circle_sum = np.sum(proposed_circle_filled)

    inside_area = overlap == proposed_circle_sum

    overlap = np.sum(np.logical_and(proposed_circle_filled, colony_binary_mask))

    has_colonies = overlap > 0

    proposed_circle_outlined = np.zeros_like(circle_mask, dtype=np.uint8)
    cv2.circle(proposed_circle_outlined, center, radius + 10, 1, 12)
    overlap = np.sum(np.logical_and(proposed_circle_outlined, colony_binary_mask))

    no_colony_overlap = overlap == 0

    return inside_area and no_colony_overlap and has_colonies

In [ ]:
import torch
from diffusers import AutoPipelineForInpainting
from diffusers.utils import make_image_grid

pipeline = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
pipeline.enable_model_cpu_offload()


def generate_patched_image(image, mask_image):
    init_image_downscaled = image.resize((1024, 1024))
    mask_image_downscaled = mask_image.resize((1024, 1024))

    # generator = torch.Generator("cuda").manual_seed(92)
    prompt = "a clean petri dish with nutrient agar medium under laboratory lighting"
    negative_prompt = (
        "bacteria colony, bacterial colonies, bacteria, circles, dots, microbial spots"
    )
    image = pipeline(
        prompt=prompt,
        strength=0.99,
        negative_prompt=negative_prompt,
        image=init_image_downscaled,
        mask_image=mask_image_downscaled,
        # num_images_per_prompt=4,
        # generator=generator,
    ).images[0]

    # Upscale the image to the original size
    image_upscaled = image.resize(init_image.size, resample=3)

    return image_upscaled

In [ ]:
import os
import csv
from PIL import Image

images = os.listdir("../training_data/images")

# read metadata.csv
metadata = []
with open("../training_data/metadata.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        metadata.append(row)

usable_images = [row[0].replace(".png", ".jpg") for row in metadata if int(row[1]) > 20]

images = [image for image in images if image in usable_images]

print(f"Number of usable images: {len(images)}")

for image in images:
    circle_mask = cv2.imread(
        "../data/labeled/circle_masks/" + image.replace(".jpg", ".png"),
        cv2.IMREAD_GRAYSCALE,
    )

    binary_mask = cv2.imread(
        "../training_data/binary_masks" + image.replace(".jpg", ".png"),
        cv2.IMREAD_GRAYSCALE,
    )

    circles = []

    for i in range(10000):
        center, radius = generate_random_circle(circle_mask.shape)
        if check_circle(center, radius, circle_mask, binary_mask):
            circles.append((center, radius))
        if len(circles) == 5:
            break

    for circle in circles:
        center, radius = circle

        proposed_circle_filled = np.zeros_like(circle_mask, dtype=np.uint8)
        cv2.circle(proposed_circle_filled, center, radius, 1, -1)

        image = Image.open("../training_data/images/" + image)
        mask_image = Image.fromarray(proposed_circle_filled)

        generate_patched_image(image, mask_image).save(
            f"../gen_data/patched_images/{image.replace('.jpg', '')}_{center[0]}_{center[1]}_{radius}.png"
        )